# Study 866 — Flight-to-Quality Beta 🛟

**Which stocks are *true* defensives — the ones that actually rally with Treasuries when
the market sells off? Do those hedges under-earn (you pay for the protection), and do
they really cushion crashes?**

For each name we estimate a **flight-to-quality beta** (`beta_ftq`): its beta to the
**TLT** long-Treasury daily return, measured **only on down-SPY days**. A high FTQ beta
is a stock that reliably co-moves with the safe-haven bid in sell-offs — a good crash
hedge. The CAPM-of-insurance prediction is two-sided: such names should (a) earn a
**lower** average return (you pay an insurance premium) yet (b) deliver **real crash
protection**. We test both on a liquid US cross-section (2010-01-04 → 2026-06-30,
50 names).

*Numbers below are the frozen headline (`docs/results.md`); the live cells run the fast
synthetic control. Survivorship: current-membership mega-caps — magnitudes are an upper
bound.*


## 1. The idea in one picture

When the market panics, investors dump stocks and pile into the safest bonds — long Treasuries rally as equities fall. A stock that *rises with that bond bid* on the worst days is a genuine hedge. The theory: hedges are expensive, so they should quietly **under-earn** in calm times — but pay you back by **losing less** when it all goes wrong. We measure each name's flight-to-quality beta and check both halves.

In [1]:
import numpy as np, pandas as pd
R = dict(spread_pct=0.52, t_nw=1.36, spread_ann_pct=6.24, lo_pct=1.79, hi_pct=1.27,
         crash_cushion=1.13, crash_welch=6.78, lo_crash=-3.25, hi_crash=-2.13)
print('pay-for-the-hedge spread (long low-FTQ / short high-FTQ):')
print('  %+.2f %%/mo  (%+.2f %%/yr)   Newey-West t = %+.2f'
      % (R['spread_pct'], R['spread_ann_pct'], R['t_nw']))
print('  low-FTQ book %+.2f vs high-FTQ book %+.2f %%/mo' % (R['lo_pct'], R['hi_pct']))
print()
print('crash protection (worst 5% of SPY days):')
print('  low-FTQ book %+.2f%% vs high-FTQ book %+.2f%% -> cushion %+.2f%%/day (t = %+.2f)'
      % (R['lo_crash'], R['hi_crash'], R['crash_cushion'], R['crash_welch']))

pay-for-the-hedge spread (long low-FTQ / short high-FTQ):
  +0.52 %/mo  (+6.24 %/yr)   Newey-West t = +1.36
  low-FTQ book +1.79 vs high-FTQ book +1.27 %/mo

crash protection (worst 5% of SPY days):
  low-FTQ book -3.25% vs high-FTQ book -2.13% -> cushion +1.13%/day (t = +6.78)


## 2. Is the sort just lucky? A live synthetic control

We plant the pay-for-the-hedge effect in a seeded toy world (`edge>0`) — names with a high flight-to-quality loading get a lower forward mean — and check the detector recovers it, while staying *silent* on the null (`edge=0`, FTQ betas present but unpriced). No network.

In [2]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
from ftq_beta import data, strategy as st
null = st.synthetic_detect(data.synthetic_panel(edge=0.0, seed=866, n_assets=40, n_days=1400), min_stocks=10)
planted = st.synthetic_detect(data.synthetic_panel(edge=0.004, seed=866, n_assets=40, n_days=1500), min_stocks=10)
print('null world   : spread NW t = %+.2f  (should be ~0)' % null['t_nw'])
print('planted world: spread NW t = %+.2f  (should light up)' % planted['t_nw'])

null world   : spread NW t = -0.21  (should be ~0)
planted world: spread NW t = +12.39  (should light up)


## 3. The honest verdict — a real hedge, a weak premium

The **crash-protection** half of the claim is **confirmed and strong**: on the worst 5% of market days the high-FTQ (hedge) book lost only -2.13% vs the low-FTQ book's -3.25% — a **+1.13%/day** cushion (Welch *t* = +6.78). FTQ beta really does pick the crash-cushioning names.

But the **pay-for-the-hedge** half is only **weak**: the long-low-FTQ / short-high-FTQ spread is **+0.52 %/mo** (the right sign — the risky names out-earned the hedges) but the Newey-West *t* is just **+1.36**, short of significance, and it holds in neither era. And it does not survive costs. **Signal: Weak · Tradability: Mirage · Crash-protection: Confirmed.**